# 🎬 VideoTransDub - One-Click Video Translation & Dubbing

**Production-grade pipeline** running on Google Colab with Streamlit UI.

### Features
- **Real ASR**: faster-whisper (Whisper small/tiny on GPU)
- **Free TTS**: Microsoft Edge-TTS (no API key needed)
- **Modern UI**: Streamlit Dark Mode dashboard with SRT editor
- **Google Drive**: Auto-sync output & checkpoints
- **Fault-tolerant**: Checkpoint/resume across disconnections

### How to use
1. **Run all cells** (Runtime → Run all)
2. Click the **Streamlit URL** that appears in Cell 4
3. Upload your video in the UI and start the pipeline

---

## Cell 1: Mount Google Drive & Clone Repository

In [ ]:
# ============================================================
# Cell 1: Google Drive Mount + Repository Setup
# ============================================================
import os
from pathlib import Path

# --- Google Drive Mount ---
GDRIVE_ROOT = Path('/content/drive')
GDRIVE_VTD = Path('/content/drive/MyDrive/VideoTransDub')

if not GDRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('✅ Google Drive mounted')
else:
    print('✅ Google Drive already mounted')

# Create persistent output directory on Drive
GDRIVE_VTD.mkdir(parents=True, exist_ok=True)
(GDRIVE_VTD / 'output').mkdir(exist_ok=True)
(GDRIVE_VTD / 'checkpoints').mkdir(exist_ok=True)
print(f'✅ Drive output dir: {GDRIVE_VTD}')

# --- Clone / update repo ---
REPO_DIR = Path('/content/pyvideotrans')
if not REPO_DIR.exists():
    !git clone https://github.com/pyvideotrans/pyvideotrans.git /content/pyvideotrans
    print('✅ Repository cloned')
else:
    %cd /content/pyvideotrans
    !git pull --ff-only 2>/dev/null || echo '⚠️ Pull skipped (local changes)'
    print('✅ Repository updated')

%cd /content/pyvideotrans
print(f'\n📂 Working directory: {os.getcwd()}')

## Cell 2: Install Dependencies (No-fail Setup)

In [ ]:
# ============================================================
# Cell 2: Environment Setup - Force Install Everything
# ============================================================
import subprocess, shutil, sys

def setup_environment():
    """No-fail environment setup for Colab."""
    errors = []

    # --- 1. System packages ---
    print('📦 Installing system packages...')
    subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
    result = subprocess.run(
        ['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libsndfile1'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        errors.append(f'apt-get failed: {result.stderr}')

    # Verify ffmpeg
    ffmpeg_path = shutil.which('ffmpeg')
    if ffmpeg_path:
        ver = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
        print(f'  ✅ ffmpeg: {ver.stdout.splitlines()[0]}')
    else:
        errors.append('ffmpeg not found after install!')
        print('  ❌ ffmpeg NOT FOUND')

    ffprobe_path = shutil.which('ffprobe')
    if ffprobe_path:
        print(f'  ✅ ffprobe: found at {ffprobe_path}')
    else:
        print('  ⚠️ ffprobe not found')

    # --- 2. Python packages ---
    print('\n🐍 Installing Python packages...')
    packages = [
        'pydantic>=2,<3',
        'PyYAML>=6,<7',
        'faster-whisper>=1.0.0',
        'edge-tts>=6.1.0',
        'opencv-python-headless>=4.8.0',
        'streamlit>=1.30.0',
        'psutil>=5.9.0',
        'ipywidgets>=8.0.0',
        'dashscope>=1.20.0',
    ]
    for pkg in packages:
        name = pkg.split('>=')[0].split('<')[0]
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--quiet', pkg],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f'  ✅ {name}')
        else:
            errors.append(f'pip install {name} failed: {result.stderr}')
            print(f'  ❌ {name}: {result.stderr[:100]}')

    # --- 3. Install videotransdub package ---
    print('\n📦 Installing videotransdub...')
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '-e',
         'apps/videotransdub'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print('  ✅ videotransdub installed')
    else:
        errors.append(f'videotransdub install failed: {result.stderr}')
        print(f'  ❌ videotransdub: {result.stderr[:200]}')

    # --- 4. Cloudflare tunnel ---
    print('\n🌐 Installing Cloudflare tunnel...')
    if not shutil.which('cloudflared'):
        subprocess.run(
            ['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb',
             '-O', '/tmp/cloudflared.deb'],
            capture_output=True
        )
        subprocess.run(['dpkg', '-i', '/tmp/cloudflared.deb'], capture_output=True)
    if shutil.which('cloudflared'):
        print('  ✅ cloudflared ready')
    else:
        print('  ⚠️ cloudflared not available (will use Colab proxy)')

    # --- 5. Runtime dirs ---
    print('\n📂 Creating runtime directories...')
    from pathlib import Path
    for d in ['runtime/workspace', 'runtime/output', 'runtime/uploads']:
        p = Path(f'apps/videotransdub/{d}')
        p.mkdir(parents=True, exist_ok=True)
    print('  ✅ Runtime dirs ready')

    # --- 6. GPU check ---
    print('\n🖥️ System info:')
    try:
        gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                              '--format=csv,noheader'], capture_output=True, text=True)
        if gpu.returncode == 0:
            print(f'  ✅ GPU: {gpu.stdout.strip()}')
        else:
            print('  ⚠️ No GPU detected (ASR will use CPU)')
    except FileNotFoundError:
        print('  ⚠️ nvidia-smi not found')

    import psutil
    mem = psutil.virtual_memory()
    print(f'  ✅ RAM: {mem.total / (1024**3):.1f} GB')
    disk = psutil.disk_usage('/')
    print(f'  ✅ Disk: {disk.free / (1024**3):.1f} GB free')

    # --- Summary ---
    if errors:
        print(f'\n⚠️ Setup completed with {len(errors)} warnings:')
        for e in errors:
            print(f'  - {e[:120]}')
    else:
        print('\n✅ Setup complete! All dependencies installed successfully.')

    return len(errors) == 0

setup_ok = setup_environment()

## Cell 3: System Dashboard (GPU / RAM / Disk)

In [ ]:
# ============================================================
# Cell 3: System Dashboard with ipywidgets
# ============================================================
import ipywidgets as widgets
from IPython.display import display, HTML
import psutil
import subprocess

def create_dashboard():
    # RAM
    mem = psutil.virtual_memory()
    ram_bar = widgets.FloatProgress(
        value=mem.percent, min=0, max=100,
        description='RAM:',
        bar_style='info' if mem.percent < 80 else 'warning',
        style={'description_width': '60px', 'bar_color': '#58a6ff'},
        layout=widgets.Layout(width='400px')
    )
    ram_label = widgets.HTML(
        f'<span style="color:#c9d1d9">{mem.used/(1024**3):.1f} / {mem.total/(1024**3):.1f} GB</span>'
    )

    # Disk
    disk = psutil.disk_usage('/')
    disk_bar = widgets.FloatProgress(
        value=disk.percent, min=0, max=100,
        description='Disk:',
        bar_style='info' if disk.percent < 85 else 'danger',
        style={'description_width': '60px', 'bar_color': '#3fb950'},
        layout=widgets.Layout(width='400px')
    )
    disk_label = widgets.HTML(
        f'<span style="color:#c9d1d9">{disk.free/(1024**3):.1f} GB free of {disk.total/(1024**3):.1f} GB</span>'
    )

    # GPU
    gpu_html = '<span style="color:#8b949e">No GPU detected</span>'
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,utilization.gpu,memory.used,memory.total',
             '--format=csv,noheader'],
            capture_output=True, text=True, timeout=5
        )
        if result.returncode == 0:
            parts = result.stdout.strip().split(',')
            name = parts[0].strip()
            util = parts[1].strip()
            mem_used = parts[2].strip()
            mem_total = parts[3].strip()
            gpu_html = f'<span style="color:#3fb950">🟢 {name} | {util} | {mem_used} / {mem_total}</span>'
    except Exception:
        pass

    gpu_widget = widgets.HTML(gpu_html)

    header = widgets.HTML(
        '<h3 style="color:#58a6ff; margin:0 0 10px 0;">📊 System Resources</h3>'
    )

    dashboard = widgets.VBox([
        header,
        widgets.HBox([ram_bar, ram_label]),
        widgets.HBox([disk_bar, disk_label]),
        widgets.HTML('<b style="color:#c9d1d9">GPU:</b>'),
        gpu_widget,
    ], layout=widgets.Layout(
        padding='15px',
        border='1px solid #30363d',
        border_radius='12px',
        width='600px',
    ))

    display(dashboard)

create_dashboard()

## Cell 4: Launch Streamlit UI with Cloudflare Tunnel

In [ ]:
# ============================================================
# Cell 4: Launch Streamlit + Cloudflare Tunnel
# ============================================================
import subprocess
import time
import shutil
import re
from IPython.display import display, HTML

STREAMLIT_PORT = 8501
APP_PATH = 'apps/videotransdub/src/videotransdub/app.py'

# Start Streamlit in background
print('🚀 Starting Streamlit server...')
st_proc = subprocess.Popen(
    ['streamlit', 'run', APP_PATH,
     '--server.port', str(STREAMLIT_PORT),
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false',
     '--theme.base', 'dark',
     '--theme.primaryColor', '#58a6ff',
     '--theme.backgroundColor', '#0e1117',
     '--theme.secondaryBackgroundColor', '#161b22',
     '--theme.textColor', '#fafafa',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(5)  # Wait for Streamlit to start
print(f'  ✅ Streamlit PID: {st_proc.pid}')

# Start Cloudflare tunnel
tunnel_url = None
if shutil.which('cloudflared'):
    print('🌐 Starting Cloudflare tunnel...')
    cf_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{STREAMLIT_PORT}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    # Wait and extract URL
    for _ in range(30):
        time.sleep(1)
        line = cf_proc.stderr.readline().decode('utf-8', errors='ignore')
        match = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
        if match:
            tunnel_url = match.group(1)
            break

    if tunnel_url:
        print(f'  ✅ Tunnel ready!')
    else:
        print('  ⚠️ Could not get tunnel URL, using Colab proxy')
else:
    print('⚠️ cloudflared not available, using Colab proxy')

# Display access URLs
colab_url = f'https://localhost:{STREAMLIT_PORT}'  # Colab internal

html = f'''
<div style="background:#161b22; border:1px solid #30363d; border-radius:12px; padding:20px; margin:10px 0;">
    <h2 style="color:#58a6ff; margin:0 0 15px 0;">🎬 VideoTransDub UI Ready!</h2>
'''
if tunnel_url:
    html += f'''
    <p style="color:#3fb950; font-size:18px; font-weight:bold;">
        🌐 Public URL: <a href="{tunnel_url}" target="_blank" style="color:#58a6ff;">{tunnel_url}</a>
    </p>
    <p style="color:#8b949e;">This URL is accessible from any device. Low latency via Cloudflare.</p>
'''
else:
    html += '''
    <p style="color:#d29922;">⚠️ No tunnel available. Use the Colab output proxy or port forwarding.</p>
'''
html += '''
    <hr style="border-color:#30363d;">
    <p style="color:#8b949e; font-size:13px;">Tip: If the UI disconnects, just reopen the URL — your pipeline state is saved in checkpoint.json.</p>
</div>
'''

display(HTML(html))

## Cell 5: End-to-End Pipeline Test (Real Engines)

This cell runs a **real** pipeline test using:
- **faster-whisper** (small model) for ASR
- **Google Translate** (free) for translation
- **Edge-TTS** for speech synthesis

It uses a short sample video to verify the full pipeline works end-to-end.

In [ ]:
# ============================================================
# Cell 5: Real End-to-End Pipeline Test
# ============================================================
import subprocess
import json
from pathlib import Path

TEST_VIDEO = Path('/content/test_sample.mp4')

# Create a short test video with speech using ffmpeg
if not TEST_VIDEO.exists():
    print('🎬 Creating test video with speech audio...')
    # Generate a 5-second test video with a tone (as placeholder)
    # In real usage, user uploads their own video
    subprocess.run([
        'ffmpeg', '-y',
        '-f', 'lavfi', '-i', 'sine=frequency=440:duration=5',
        '-f', 'lavfi', '-i', 'color=c=black:s=640x360:d=5',
        '-c:v', 'libx264', '-c:a', 'aac',
        '-shortest',
        str(TEST_VIDEO)
    ], capture_output=True)
    print(f'  ✅ Test video: {TEST_VIDEO} ({TEST_VIDEO.stat().st_size:,} bytes)')

# Run the real pipeline with real_free preset
print('\n🚀 Running real pipeline (real_free preset)...')
print('  ASR: faster-whisper (small)')
print('  Translation: Google Free')
print('  TTS: Edge-TTS')
print('  Target: Vietnamese\n')

result = subprocess.run(
    ['python3', '-m', 'videotransdub.cli', 'run',
     '--config', 'apps/videotransdub/configs/default.yaml',
     '--config', 'apps/videotransdub/configs/presets/real_free.yaml',
     '--video-path', str(TEST_VIDEO),
     '--target-language', 'vi',
    ],
    capture_output=True, text=True, timeout=600,
    env={**dict(__import__('os').environ), 'PYTHONPATH': 'apps/videotransdub/src'},
)

print('--- STDOUT ---')
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('\n--- STDERR ---')
    print(result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)
    print(f'\n❌ Pipeline failed (exit code {result.returncode})')
else:
    print('\n✅ Pipeline completed successfully!')
    # Show output
    workspace_dirs = list(Path('apps/videotransdub/runtime/workspace').iterdir())
    if workspace_dirs:
        ws = sorted(workspace_dirs)[-1]
        manifest = ws / 'manifests' / 'job.json'
        if manifest.exists():
            data = json.loads(manifest.read_text())
            print(f'\n📋 Job: {data.get("job_id")}')
            for stage_name, stage_info in data.get('stages', {}).items():
                print(f'  {stage_name}: {stage_info.get("status", "unknown")}')
            final = data.get('artifacts', {}).get('final_video', '')
            if final and Path(final).exists():
                print(f'\n🎥 Output: {final} ({Path(final).stat().st_size:,} bytes)')

## Cell 6: Sync Output to Google Drive

In [ ]:
# ============================================================
# Cell 6: Sync workspace & output to Google Drive
# ============================================================
import shutil
from pathlib import Path

WORKSPACE_DIR = Path('apps/videotransdub/runtime/workspace')
GDRIVE_VTD = Path('/content/drive/MyDrive/VideoTransDub')

if GDRIVE_VTD.exists():
    print('📁 Syncing to Google Drive...')
    for ws in WORKSPACE_DIR.iterdir():
        if not ws.is_dir():
            continue
        dest = GDRIVE_VTD / 'checkpoints' / ws.name
        # Sync checkpoint and manifest
        manifest_dir = ws / 'manifests'
        if manifest_dir.exists():
            dest_m = dest / 'manifests'
            dest_m.mkdir(parents=True, exist_ok=True)
            for f in manifest_dir.iterdir():
                shutil.copy2(f, dest_m / f.name)

        # Sync output
        output_dir = ws / 'output'
        if output_dir.exists():
            for f in output_dir.iterdir():
                if f.suffix in ('.mp4', '.mkv', '.json'):
                    shutil.copy2(f, GDRIVE_VTD / 'output' / f.name)
                    print(f'  ✅ {f.name} -> Drive')

    print('\n✅ Sync complete!')
    print(f'   Output: {GDRIVE_VTD / "output"}')
    print(f'   Checkpoints: {GDRIVE_VTD / "checkpoints"}')
else:
    print('⚠️ Google Drive not mounted. Run Cell 1 first.')

## Cell 7: CLI Mode (Alternative to UI)

Use this cell to run the pipeline directly via CLI without the Streamlit UI.

In [ ]:
# ============================================================
# Cell 7: Direct CLI execution (alternative to UI)
# ============================================================

# --- Configure these ---
VIDEO_PATH = '/content/drive/MyDrive/your_video.mp4'  # Change this!
PRESET = 'real_free'  # Options: real_free, fast_free, qwen_free, balanced, quality_api, mock
TARGET_LANG = 'vi'
SOURCE_LANG = 'auto'

# For qwen_free preset, set your API key:
import os
os.environ['QWEN_API_KEY'] = 'sk-abd0a732837b414696fc23a8f933aa3b'

# --- Run ---
!PYTHONPATH=apps/videotransdub/src python3 -m videotransdub.cli run \
    --config apps/videotransdub/configs/default.yaml \
    --config apps/videotransdub/configs/presets/{PRESET}.yaml \
    --video-path "{VIDEO_PATH}" \
    --target-language {TARGET_LANG} \
    --source-language {SOURCE_LANG}